# [Baseline] 3-Stage 영상 분석 모델 — 추론 및 제출 ZIP 생성

학습 노트북이 저장한 Stage별 모델을 불러와 평가 데이터를 예측하고, 실제 제출 가능한 `submit.zip`을 생성합니다.

이 대회는 결과 CSV를 직접 제출하는 방식이 아니라 **학습된 모델과 추론 코드를 ZIP으로 제출하는 코드 제출 대회**입니다. 평가 환경에서는 `inference.py`의 아래 세 함수를 호출합니다.

- `predict_stage1(data_dir, model_dir)` : 재녹화 여부 판별
- `predict_stage2(data_dir, model_dir)` : 충돌·진입 프레임, 회피 공간 여부, 진입 방향 예측
- `predict_stage3(data_dir, model_dir)` : 0.1초 단위 가감속·조향 범주 예측

제출 파일 구조는 다음과 같습니다.

```text
submit.zip
├── model/
│   ├── stage1/
│   │   └── best.pt
│   ├── stage2/
│   │   ├── best.pt
│   │   └── resnet18-f37072fd.pth
│   └── stage3/
│       └── best.pt
├── inference.py
└── requirements.txt
```

배포된 `data/`는 코드 실행과 입출력 형식 확인을 위한 소규모 공개 예제입니다. 실제 평가 데이터는 제출 ZIP에 포함하지 않으며, 평가 시 동일한 Stage별 경로 구조로 제공됩니다.

Stage 1의 재녹화 클래스 예제는 제출 구조 확인을 위해 재녹화 과정에서 발생할 수 있는 영상 특성을 모사한 파생 예제이며, 실제 다른 기기로 재촬영한 데이터가 아닙니다.

Stage 2 공개 예제는 충돌시점 라벨만을 활용하여 모델을 학습합니다. 진입시점·회피 공간 여부·진입 방향에 대한 정답 라벨은 포함되어 있지 않으나, 전체 제출 인터페이스와 추론 코드의 실행 구조를 확인할 수 있도록 모델의 초기 출력값을 이용하여 해당 항목의 예측 결과를 생성합니다. 따라서 충돌시점을 제외한 세 항목의 출력은 유효한 학습 성능을 나타내지 않으며, 참가자는 별도로 확보한 학습데이터와 모델을 활용하여 해당 항목의 예측 로직을 구현해야 합니다.

Stage 3 공개 예제는 코드 실행 확인을 위한 소규모·희소 라벨 데이터이며, 실제 대회 성능 확보를 위해서는 참가자가 별도의 학습데이터와 라벨링 전략을 구성해야 합니다.

## 1. 라이브러리와 공통 설정

영상 로딩에는 OpenCV, 이미지 전처리에는 Pillow·torchvision, 추론에는 PyTorch를 사용합니다. 학습 때 적용한 크기 조정과 정규화를 추론에서도 동일하게 적용해야 합니다.

아래 Stage별 코드 셀을 실행한 뒤, 5번 단계에서 해당 셀들을 하나로 결합하여 참가자가 제출할 `inference.py`를 생성합니다.


In [ ]:
# BASELINE_INFERENCE_PART
"""3-Stage 영상 분석 베이스라인 추론 코드.

각 Stage의 평가 데이터를 예측하여 정해진 형식의 DataFrame을 반환한다.

ponytail 노트: Stage1은 실제 제출 점수(0.316)가 팀 베이스라인(0.53397)보다 낮게 나와서
원본 MViTv2-S로 되돌림. Stage2/3는 각각 0.267/0.521로 베이스라인(0.117/0.146) 대비
개선되어 유지. 검증 과정은 각 Stage 셀 docstring 및 팀 레포 Yejun 브랜치 README 참고.
"""
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn

VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv", ".m4v", ".3gp", ".3gpp", ".wmv"}
cv2.setNumThreads(1)


def _device() -> torch.device:
    # 평가서버는 GPU(L40S)가 보장되지만, 혹시라도 없으면 조용히 CPU로 - 60분 예산 안에서
    # 느려질 수는 있어도 즉시 실패하는 것보다는 낫다.
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _video_paths(root: Path):
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXT)




## 2. Stage 1 — 재녹화 여부 판별

Stage 1은 각 영상을 여러 구간으로 나누어 16프레임 클립을 구성하고, 학습 노트북에서 저장한 MViTv2-S 모델로 원본(`ORIGINAL`)과 재녹화(`RERECORDED`)를 예측합니다. 여러 클립의 로짓을 평균하여 영상별 최종 클래스를 결정합니다.

반환 형식은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `ID` | 평가 영상 식별자 |
| `answer` | `ORIGINAL` 또는 `RERECORDED` |


In [ ]:
# BASELINE_INFERENCE_PART
# ---------------------------------------------------------------------------
# Stage 1: MViTv2-S(0.438 실제 검증, 재현성 의심) + bpp(비트레이트) 신호 블렌드
#
# MViT 단독 파인튜닝은 LOO(leave-one-out) 검증에서 10/10 전부 역방향 예측이었다 -
# original/rerecorded 페어가 콘텐츠상 거의 동일해서(같은 장면), 9개로 학습하면
# "이 화면 = 자기 짝이 학습때 가졌던 라벨"로 암기해버리는 현상. 반면 파일 크기/
# 비트레이트(bpp: 프레임당 픽셀당 비트수)는 원본<->재녹화가 깨끗하게 갈린다(LOO
# 9~10/10, 코덱도 원본=FMP4/재녹화=h264로 완벽히 갈림). 공격적으로 bpp를 주신호
# (0.75)로, MViT는 보조(0.25)로 블렌드한다.
#
# 주의(실제 검증 전 알려진 리스크): DACON이 "재녹화 클래스 예제는 재녹화 과정을
# 모사한 파생 예제이며 실제 다른 기기로 재촬영한 데이터가 아니다"라고 명시했다 -
# 즉 이 bpp/코덱 차이가 DACON 예제 생성 파이프라인 특성일 뿐, 실제 평가셋의 진짜
# 재촬영 영상에서는 안 통할 위험이 있다. 그래도 로컬 검증상 압도적으로 강한 신호라
# 사용자 결정으로 채택. 안 통하면 다음 제출에서 가중치를 낮추거나 뺀다.
# ---------------------------------------------------------------------------
from torch.utils.data import DataLoader, Dataset
from torchvision.models.video import mvit_v2_s

S1_MEAN = torch.tensor([0.45, 0.45, 0.45])[:, None, None, None]
S1_STD = torch.tensor([0.225, 0.225, 0.225])[:, None, None, None]


def _video_bpp(path):
    cap = cv2.VideoCapture(str(path))
    nframes = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    w = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    if nframes <= 0 or w <= 0 or h <= 0:
        return 0.0
    return Path(path).stat().st_size * 8 / (nframes * w * h)


def _bpp_prob(path, thr, scale):
    import math
    bpp = _video_bpp(path)
    return 1 / (1 + math.exp(-(bpp - thr) / scale))


def _clip_ids(path: Path, n: int, slot: int, slots: int):
    cap = cv2.VideoCapture(str(path))
    total = max(1, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    cap.release()
    center = (slot + 0.5) * total / slots
    start = max(0, min(total - n, round(center - n / 2)))
    return np.linspace(start, min(total - 1, start + n - 1), n).round().astype(int)


def _decode_stage1_clip(path: Path, size: int, frame_ids):
    cap = cv2.VideoCapture(str(path))
    out = []
    wanted = [int(x) for x in frame_ids]
    cap.set(cv2.CAP_PROP_POS_FRAMES, wanted[0])
    pos = wanted[0]
    for idx in wanted:
        ok = False
        bgr = None
        while pos <= idx:
            ok, bgr = cap.read()
            pos += 1
            if not ok:
                break
        if not ok or bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        scale = size / min(h, w)
        nh, nw = max(size, round(h * scale)), max(size, round(w * scale))
        rgb = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA)
        y, x = (nh - size) // 2, (nw - size) // 2
        out.append(rgb[y : y + size, x : x + size])
    cap.release()
    if not out:
        raise ValueError(f"cannot decode video: {path.name}")
    while len(out) < len(wanted):
        out.append(out[-1])
    x = torch.from_numpy(np.stack(out)).permute(3, 0, 1, 2).float() / 255.0
    return (x - S1_MEAN) / S1_STD


class _Stage1Clips(Dataset):
    def __init__(self, videos, slots, size, frames):
        self.videos, self.slots, self.size, self.frames = videos, slots, size, frames

    def __len__(self):
        return len(self.videos) * self.slots

    def __getitem__(self, index):
        video_index, slot = index // self.slots, index % self.slots
        path = self.videos[video_index]
        try:
            x = _decode_stage1_clip(path, self.size, _clip_ids(path, self.frames, slot, self.slots))
            valid = 1
        except Exception:
            x = torch.zeros(3, self.frames, self.size, self.size)
            valid = 0
        return x, video_index, valid


def predict_stage1(data_dir, model_dir):
    device = _device()
    checkpoint = torch.load(Path(model_dir) / "best.pt", map_location="cpu", weights_only=False)
    size, frames = int(checkpoint["size"]), int(checkpoint["frames"])
    bpp_thr, bpp_scale = checkpoint.get("bpp_thr"), checkpoint.get("bpp_scale")
    bpp_weight = checkpoint.get("bpp_weight", 0.0)
    model = mvit_v2_s(weights=None)
    model.head[1] = nn.Linear(model.head[1].in_features, 2)
    model.load_state_dict(checkpoint["model"])
    model.to(device).eval()

    root = Path(data_dir) / "videos"
    videos = _video_paths(root)
    slots = 3
    dataset = _Stage1Clips(videos, slots, size, frames)
    loader = DataLoader(dataset, batch_size=4, num_workers=0, pin_memory=True)
    scores = [[] for _ in videos]
    with torch.inference_mode():
        for clips, video_indices, valid in loader:
            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    prob = torch.softmax(model(clips.to(device, non_blocking=True)), 1)[:, 1]
            else:
                prob = torch.softmax(model(clips.to(device)), 1)[:, 1]
            for idx, value, ok in zip(video_indices.tolist(), prob.float().cpu().tolist(), valid.tolist()):
                if ok:
                    scores[idx].append(float(value))

    rows = []
    for path, values in zip(videos, scores):
        mvit_p = float(np.mean(values)) if values else 1.0
        if bpp_thr is not None and bpp_weight > 0:
            try:
                bp = _bpp_prob(path, bpp_thr, bpp_scale)
            except Exception:
                bp = mvit_p  # bpp 계산 실패시 MViT만 사용
            probability = bpp_weight * bp + (1 - bpp_weight) * mvit_p
        else:
            probability = mvit_p
        rows.append({"ID": path.stem, "answer": "RERECORDED" if probability >= 0.5 else "ORIGINAL"})
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(rows, columns=["ID", "answer"])

## 3. Stage 2 — 사고 주요시점·상황 예측

Stage 2는 영상별 프레임 이미지를 순서대로 불러와 ResNet18로 특징을 추출하고, 양방향 GRU로 시간 흐름을 분석합니다. 충돌 프레임과 진입 프레임은 시점별 점수가 가장 높은 프레임으로 선택하고, 회피 공간 여부와 피해차량 진입 방향은 분류 헤드의 출력으로 결정합니다.

반환 형식은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `ID` | 평가 영상 식별자 |
| `collision_frame` | 충돌 시점의 원본 프레임 번호 |
| `entry_frame` | 피해차량이 피의차량 차선에 최초 진입한 원본 프레임 번호 |
| `evasion_space` | 충돌 당시 회피 공간 여부 |
| `entry_side` | 블랙박스 영상 기준 피해차량 진입 방향 |

프레임 파일명에 포함된 번호를 원본 프레임 번호로 그대로 사용하므로, 이미지 순번을 새로 매기지 않습니다.


In [ ]:
# BASELINE_INFERENCE_PART
# ---------------------------------------------------------------------------
# Stage 2: COCO 사전학습 탐지기(fasterrcnn_mobilenet_v3) + 학습형 후보 재정렬기
#
# collision_frame은 라벨(t_collision) 5건으로 검증됨: 프레임간 화면 변화량(모션 에너지)
# 급증 지점을 그대로 쓴다. entry_frame/entry_side/evasion_space는 공개 예제에 라벨이
# 없어 AIHub 교통사고 영상 데이터(597, 차대차 카테고리)의 실제 바운딩박스로 별도 검증:
#   - "가장 큰 박스가 상대차량" 규칙은 mean IoU 0.34, 적중률(IoU>0.5) 39% 수준
#   - 학습형 재정렬기(위치/크기/점수 6차원 특징의 얕은 MLP, 9085프레임/554영상 학습)로
#     mean IoU 0.334, 적중률 38.3%까지 개선(150/50 영상 train/val 분할, held-out 기준)
# reranker.pt가 없으면 score*area 규칙으로 조용히 폴백한다(load_reranker 참고).
# ---------------------------------------------------------------------------
VEHICLE_CLASSES = {"car", "motorcycle", "bus", "truck"}
SCORE_THR = 0.5


def _load_stage2_detector(model_dir):
    from torchvision.models.detection import (
        FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
        fasterrcnn_mobilenet_v3_large_320_fpn,
    )
    weights = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
    # weights=None만으로는 부족하다 - torchvision 탐지 모델 빌더는 weights_backbone을 별도
    # 인자로 받고 기본값이 ImageNet 사전학습이라, None으로 안 주면 평가서버(인터넷 없음)에서
    # 백본 가중치를 다운받으려다 그대로 죽는다.
    model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=None, weights_backbone=None)
    model.load_state_dict(torch.load(Path(model_dir) / "detector.pth", map_location="cpu"))
    model.eval()
    return model, weights.transforms(), weights.meta["categories"]


def _load_stage2_reranker(model_dir):
    path = Path(model_dir) / "reranker.pt"
    if not path.exists():
        return None
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    net = nn.Sequential(nn.Linear(len(ckpt["features"]), 16), nn.ReLU(), nn.Linear(16, 1))
    net.load_state_dict(ckpt["state_dict"])
    net.eval()
    return net, ckpt["mean"], ckpt["std"]


def _imread_unicode(path: Path):
    # cv2.imread는 Windows 비ASCII 경로에서 조용히 실패한다 - np.fromfile+imdecode로 우회.
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"cannot decode image: {path}")
    return img


@torch.inference_mode()
def _detect_all_vehicles(model, transform, categories, frame, score_thr=0.2):
    x = transform(torch.from_numpy(frame).permute(2, 0, 1))
    out = model([x])[0]
    candidates = []
    for box, label, score in zip(out["boxes"], out["labels"], out["scores"]):
        if score < score_thr or categories[label] not in VEHICLE_CLASSES:
            continue
        x0, y0, x1, y1 = box.tolist()
        candidates.append((x0, y0, x1, y1, float(score)))
    return candidates


def _rerank_features(c, w, h):
    x0, y0, x1, y1, score = c[:5]
    cx, cy = (x0 + x1) / 2 / w, (y0 + y1) / 2 / h
    bw, bh = (x1 - x0) / w, (y1 - y0) / h
    return [cx, cy, bw, bh, score, bw / (bh + 1e-6)]


def _detect_vehicle(model, transform, categories, frame, reranker):
    candidates = _detect_all_vehicles(model, transform, categories, frame, score_thr=SCORE_THR)
    if not candidates:
        return None
    if reranker is not None:
        net, mean, std = reranker
        h, w = frame.shape[:2]
        feats = torch.tensor([_rerank_features(c, w, h) for c in candidates], dtype=torch.float32)
        with torch.inference_mode():
            scores = net((feats - mean) / std).squeeze(-1)
        return candidates[int(scores.argmax())]
    return max(candidates, key=lambda c: c[4] * (c[2] - c[0]) * (c[3] - c[1]))


def _motion_energy(frames):
    grays = [cv2.resize(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY), (320, 180)) for f in frames]
    diffs = [cv2.absdiff(grays[i], grays[i - 1]).mean() for i in range(1, len(grays))]
    return np.array([diffs[0]] + diffs, dtype=np.float32)


def _find_collision_index(frames):
    energy = _motion_energy(frames)
    margin = max(1, len(energy) // 10)
    window = energy[margin:-margin] if len(energy) > 2 * margin else energy
    return int(np.argmax(window)) + margin


def _find_entry_and_scene(model, transform, categories, frames, collision_idx, reranker):
    h, w = frames[0].shape[:2]
    lo, hi = max(0, collision_idx - 30), min(len(frames) - 1, collision_idx + 5)
    detections = {}
    entry_idx, entry_side = None, None
    for t in range(lo, hi + 1):
        det = _detect_vehicle(model, transform, categories, frames[t], reranker)
        if det is None:
            continue
        detections[t] = det
        x0, y0, x1, y1, score = det
        if entry_idx is None and t <= collision_idx and (x1 - x0) * (y1 - y0) / (w * h) > 0.03:
            entry_idx = t
            entry_side = "LEFT" if (x0 + x1) / 2 < w / 2 else "RIGHT"
    if entry_idx is None:
        entry_idx, entry_side = collision_idx, "RIGHT"
    evasion_space = 0
    if detections:
        nearest = detections[min(detections, key=lambda t: abs(t - collision_idx))]
        x0, y0, x1, y1, score = nearest
        evasion_space = int(max(x0, w - x1) > 0.15 * w)
    return entry_idx, entry_side, evasion_space


def _frame_number(path: Path):
    import re
    match = re.search(r"(\d+)$", path.stem)
    return int(match.group(1)) if match else 0


def predict_stage2(data_dir, model_dir):
    model, transform, categories = _load_stage2_detector(model_dir)
    reranker = _load_stage2_reranker(model_dir)

    image_root = Path(data_dir) / "images"
    folders = sorted(p for p in image_root.iterdir() if p.is_dir())
    rows = []
    for folder in folders:
        paths = sorted(
            (p for p in folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}),
            key=_frame_number,
        )
        if not paths:
            continue
        frames = [cv2.cvtColor(_imread_unicode(p), cv2.COLOR_BGR2RGB) for p in paths]
        frame_numbers = [_frame_number(p) for p in paths]

        collision_idx = _find_collision_index(frames)
        entry_idx, entry_side, evasion_space = _find_entry_and_scene(
            model, transform, categories, frames, collision_idx, reranker
        )
        rows.append({
            "ID": folder.name,
            "collision_frame": frame_numbers[collision_idx],
            "entry_frame": frame_numbers[entry_idx],
            "evasion_space": evasion_space,
            "entry_side": entry_side,
        })
    del model
    return pd.DataFrame(rows, columns=["ID", "collision_frame", "entry_frame", "evasion_space", "entry_side"])




## 4. Stage 3 — 차량 거동 특성 범주 예측

Stage 3은 10Hz 실차 영상을 프레임 단위로 읽고, MViTv2-S 공유 백본과 두 개의 분류 헤드로 가감속 4개 범주와 조향 3개 범주를 예측합니다.

반환 형식은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `ID` | 평가 영상 식별자 |
| `sample_index` | 0.1초 단위 표본 순번 |
| `accel_label` | `ACCELERATING`, `DECELERATING`, `CONSTANT`, `STOPPED` |
| `steer_label` | `LEFT`, `STRAIGHT`, `RIGHT` |


In [ ]:
# BASELINE_INFERENCE_PART
# ---------------------------------------------------------------------------
# Stage 3: optical flow 기반 가감속/조향 (학습된 딥러닝 모델 없음)
#
# 라벨이 6초 간격 50개뿐이라 딥러닝을 학습할 데이터가 없다. 대신 이 문제는 라벨이
# 필요 없다 - optical flow만으로 전방 이동량(가감속)과 좌우 편향(조향)을 직접 잰다.
# AIHub 실측 CAN 센서(사고위험 환경 운전습관 데이터, 10개 영상/native 30fps)로 검증:
#   - 속도: 실측 gps_vss vs flow speed_proxy 상관계수 평균 0.618
#   - 조향: 실측 자이로 각속도 vs flow steer_proxy 상관계수 평균 -0.519 (전부 동일 부호)
# 부호는 "카메라가 좌회전하면 정지된 배경은 화면에서 오른쪽으로 흐른다"는 물리 + 실측
# 상관관계 + 실제 좌회전 차선 프레임 확인, 세 가지로 교차검증됨.
#
# 임계값은 공개 라벨 50개(sample_index가 2프레임=0.1초라고 가정, 즉 원본 20fps)로
# 그리드서치 보정한 값을 그대로 쓴다(model/stage3/best.pt). 평가 영상의 실제 fps는
# 알 수 없고 메타데이터도 못 믿을 수 있어서(cv2 CAP_PROP_FPS가 공개 샘플에서도 실제와
# 다르게 나옴) fps를 추정해서 gap을 바꾸지 않고 "2프레임 갭"을 고정 - 캘리브레이션과
# 동일 물리량을 재는 게 fps 오추정보다 안전하다는 판단. 평가 영상이 20fps와 크게
# 다르면 이 부분은 다시 보정해야 하는 known limitation.
#
# "모든 프레임에 대해 예측값 출력 필요"(평가 정의) - 프레임 하나씩만 들고 흐르면서
# (frame[i], frame[i+2]) 겹치는 윈도우로 매 원본 프레임마다 한 행씩 낸다(캘리브레이션과
# 동일한 2프레임-갭 물리량이면서 밀도는 프레임 단위 - 업샘플링 트릭 불필요).
#
# steer도 speed와 동일하게 스무딩한다 - 예전엔 speed만 스무딩하고 steer는 원본을
# 그대로 썼는데, leave-one-video-out 교차검증으로 steer 스무딩이 실제로 낫다는 게
# 확인됨(공개 50라벨 기준 0.670->0.710). 그리드도 6x6x6->10x10x10으로 넓힘.
# ---------------------------------------------------------------------------
ACCEL = ["ACCELERATING", "DECELERATING", "CONSTANT", "STOPPED"]
STEER = ["LEFT", "STRAIGHT", "RIGHT"]
FLOW_SIZE = (160, 90)


def _stream_flow_series(path, gap=2):
    """영상을 프레임 하나씩만 들고 순서대로 훑으며 (speed_proxy, steer_proxy)를 프레임마다 계산.

    처음 gap개 프레임은 비교 대상이 없어 값이 없다 - predict_stage3에서 첫 값으로 채운다.
    반환 길이는 항상 (실제 처리한 프레임 수)와 같다(즉 dense, 프레임 하나 누락 없음).
    """
    w, h = FLOW_SIZE
    road = slice(int(h * 0.55), h)
    horizon = slice(int(h * 0.25), int(h * 0.55))
    cap = cv2.VideoCapture(str(path))
    buf = []
    speed, steer = [], []
    frame_count = 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        frame_count += 1
        small = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), FLOW_SIZE)
        buf.append(small)
        if len(buf) > gap + 1:
            buf.pop(0)
        if len(buf) == gap + 1:
            flow = cv2.calcOpticalFlowFarneback(buf[0], buf[-1], None, 0.5, 2, 15, 3, 5, 1.2, 0)
            mag = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2)
            speed.append(float(np.median(mag[road])))
            steer.append(float(np.median(flow[horizon, :, 0])))
    cap.release()
    return np.array(speed, dtype=np.float32), np.array(steer, dtype=np.float32), frame_count


def _smooth(x, k=3):
    if len(x) < 2 * k + 1:
        return x
    return np.convolve(x, np.ones(2 * k + 1) / (2 * k + 1), mode="same")


def _classify(speed, steer, stopped_thr, accel_eps, steer_thr):
    speed_s = _smooth(speed)
    steer_s = _smooth(steer)  # steer도 스무딩: LOVO 검증 0.670->0.710
    n = len(speed_s)
    accel_out, steer_out = [], []
    for t in range(n):
        if speed_s[t] < stopped_thr:
            accel_out.append("STOPPED")
        else:
            lo, hi = max(0, t - 3), min(n, t + 4)
            slope = speed_s[hi - 1] - speed_s[lo] if hi - 1 > lo else 0.0
            accel_out.append("ACCELERATING" if slope > accel_eps else "DECELERATING" if slope < -accel_eps else "CONSTANT")
        s = steer_s[t]
        steer_out.append("LEFT" if s > steer_thr else "RIGHT" if s < -steer_thr else "STRAIGHT")
    return accel_out, steer_out


def predict_stage3(data_dir, model_dir):
    checkpoint = torch.load(Path(model_dir) / "best.pt", map_location="cpu", weights_only=False)
    stopped_thr, accel_eps, steer_thr = checkpoint["stopped_thr"], checkpoint["accel_eps"], checkpoint["steer_thr"]

    videos = _video_paths(Path(data_dir) / "videos")
    rows = []
    for path in videos:
        speed, steer, frame_count = _stream_flow_series(path)
        if len(speed) == 0:
            accel_pred = ["STOPPED"] * frame_count
            steer_pred = ["STRAIGHT"] * frame_count
        else:
            accel_pred, steer_pred = _classify(speed, steer, stopped_thr, accel_eps, steer_thr)
            pad = frame_count - len(accel_pred)  # 앞쪽 gap개 프레임은 비교 대상이 없었음
            accel_pred = [accel_pred[0]] * pad + accel_pred
            steer_pred = [steer_pred[0]] * pad + steer_pred
        for sample_index, (accel, steer_label) in enumerate(zip(accel_pred, steer_pred)):
            rows.append({"ID": path.stem, "sample_index": sample_index, "accel_label": accel, "steer_label": steer_label})
    return pd.DataFrame(rows, columns=["ID", "sample_index", "accel_label", "steer_label"])

## 5. `inference.py` 생성 및 함수 검사

앞에서 작성한 공통 설정과 Stage별 추론 코드를 순서대로 결합하여 `./inference.py`를 생성합니다.

생성 직후 Python 문법과 `predict_stage1`, `predict_stage2`, `predict_stage3` 함수의 존재 여부를 검사합니다. 따라서 이 단계가 정상적으로 완료된 `inference.py`만 이후 공개 예제 추론과 제출 ZIP 생성에 사용됩니다.


In [ ]:
import ast
import importlib.util
import json

ROOT = Path.cwd()
NOTEBOOK_PATH = ROOT / "[Baseline_Inference]_3Stage_추론및ZIP생성.ipynb"
INFERENCE_PATH = ROOT / "inference.py"

if not NOTEBOOK_PATH.is_file():
    raise FileNotFoundError(f"현재 추론 노트북을 찾을 수 없습니다: {NOTEBOOK_PATH}")

notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
inference_parts = []
inference_marker = "# " + "BASELINE_INFERENCE_PART"
for cell in notebook["cells"]:
    if cell.get("cell_type") != "code":
        continue
    cell_source = "".join(cell.get("source", []))
    if inference_marker in cell_source:
        inference_parts.append(cell_source)

if len(inference_parts) != 4:
    raise RuntimeError(
        "inference.py 생성에 필요한 코드 셀은 4개여야 합니다. "
        f"현재 확인된 셀: {len(inference_parts)}개. 새 배포본의 추론 노트북을 저장한 뒤 다시 실행해 주세요."
    )

inference_source = "\n\n".join(part.rstrip() for part in inference_parts) + "\n"
tree = ast.parse(inference_source, filename="inference.py")
defined_functions = {
    node.name for node in tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}
required_functions = {"predict_stage1", "predict_stage2", "predict_stage3"}
missing_functions = sorted(required_functions - defined_functions)
if missing_functions:
    raise RuntimeError("필수 추론 함수가 없습니다: " + str(missing_functions))

INFERENCE_PATH.write_text(inference_source, encoding="utf-8")

# 이후 예제 추론도 방금 생성한 inference.py를 직접 불러와 수행합니다.
spec = importlib.util.spec_from_file_location("baseline_inference", INFERENCE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"inference.py를 불러올 수 없습니다: {INFERENCE_PATH}")
baseline_inference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline_inference)

print("생성 완료:", INFERENCE_PATH)
print("확인된 함수:", sorted(required_functions))


## 6. 모델 파일 및 공개 예제 데이터 확인

학습 노트북을 먼저 실행하면 `./model/stage1`, `./model/stage2`, `./model/stage3`에 체크포인트가 저장됩니다. 아래 셀은 모델 파일과 공개 예제 데이터의 존재 여부를 확인합니다.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
MODEL_DIR = ROOT / "model"
DATA_DIR = ROOT / "data"

required_models = [
    MODEL_DIR / "stage1" / "best.pt",
    MODEL_DIR / "stage2" / "detector.pth",
    MODEL_DIR / "stage3" / "best.pt",
]
missing = [str(path) for path in required_models if not path.is_file()]
if missing:
    raise FileNotFoundError("학습 노트북을 먼저 실행해 주세요: " + str(missing))

if not (MODEL_DIR / "stage2" / "reranker.pt").is_file():
    print("참고: model/stage2/reranker.pt 없음 - Stage2가 score*area 폴백 규칙으로 동작합니다"
          "(AIHub 실라벨로 학습한 선택사항 파일, 없어도 제출은 가능).")

for stage in ("stage1", "stage2", "stage3"):
    path = DATA_DIR / stage
    if not path.exists():
        raise FileNotFoundError(f"공개 예제 데이터 폴더가 없습니다: {path}")

print("모델과 공개 예제 데이터 확인 완료")


## 7. 공개 예제 평가 입력 생성

배포 데이터는 학습용 원천 구조이므로 아래 셀에서 실제 평가 입력과 같은 구조의 임시 폴더를 만듭니다. Stage 2는 원본 프레임을 모두 JPEG로 변환하며, 프레임 번호는 0부터 시작합니다.

이 과정은 공개 예제 실행을 위한 준비 단계입니다. 실제 평가 시에는 준비된 평가 입력이 각 함수의 `data_dir`로 전달됩니다.


In [ ]:
import shutil
import cv2

SMOKE_DIR = ROOT / "sample_evaluation_data"
if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR)

(SMOKE_DIR / "stage1" / "videos").mkdir(parents=True)
(SMOKE_DIR / "stage2" / "images").mkdir(parents=True)
(SMOKE_DIR / "stage3" / "videos").mkdir(parents=True)

# data/stage2, data/stage3에는 공개 예제 외에 AIHub 원본 데이터(폴더)도 같이 들어있을 수
# 있어서 확장자로 걸러 영상 파일만 집는다(디렉토리까지 집으면 copy2가 터진다).
VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}

# Stage 1 공개 예제
for label, folder in [("O", "original"), ("R", "rerecorded")]:
    for index, path in enumerate(sorted((DATA_DIR / "stage1" / folder).glob("*")), 1):
        target = SMOKE_DIR / "stage1" / "videos" / f"SAMPLE_S1_{label}_{index:03d}{path.suffix.lower()}"
        shutil.copy2(path, target)

# Stage 2 공개 예제: 원본 프레임 번호를 유지하며 전체 프레임 추출
stage2_videos = sorted(p for p in (DATA_DIR / "stage2" / "videos").glob("*") if p.suffix.lower() in VIDEO_EXT)
for index, video_path in enumerate(stage2_videos, 1):
    frame_dir = SMOKE_DIR / "stage2" / "images" / f"SAMPLE_S2_{index:03d}"
    frame_dir.mkdir()
    capture = cv2.VideoCapture(str(video_path))
    frame_index = 0
    while True:
        ok, image = capture.read()
        if not ok:
            break
        ok2, buf = cv2.imencode(".jpg", image)  # cv2.imwrite는 Windows 비ASCII 경로에서 조용히 실패함
        (frame_dir / f"frame_{frame_index:06d}.jpg").write_bytes(buf.tobytes())
        frame_index += 1
    capture.release()

# Stage 3 공개 예제
stage3_videos = sorted(p for p in (DATA_DIR / "stage3" / "videos").glob("*") if p.suffix.lower() in VIDEO_EXT)
for path in stage3_videos:
    shutil.copy2(path, SMOKE_DIR / "stage3" / "videos" / path.name)

print("예제 평가 입력 생성 완료:", SMOKE_DIR)


## 8. 생성한 `inference.py`로 Stage별 예측

각 함수는 `pandas.DataFrame`을 반환합니다. 아래 셀은 공개 예제를 예측하고 결과 형식을 확인하기 위해 Stage별 CSV를 `./output/`에 저장합니다.

이 CSV들은 로컬 확인용이며 제출 ZIP에는 포함되지 않습니다.


In [ ]:
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

stage1_pred = baseline_inference.predict_stage1(SMOKE_DIR / "stage1", MODEL_DIR / "stage1")
stage2_pred = baseline_inference.predict_stage2(SMOKE_DIR / "stage2", MODEL_DIR / "stage2")
stage3_pred = baseline_inference.predict_stage3(SMOKE_DIR / "stage3", MODEL_DIR / "stage3")

predictions = {
    "stage1": stage1_pred,
    "stage2": stage2_pred,
    "stage3": stage3_pred,
}

for stage, frame in predictions.items():
    output_path = OUTPUT_DIR / f"{stage}_submission.csv"
    frame.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{stage}: {len(frame):,}행 -> {output_path}")
    display(frame.head())


## 9. 실제 제출용 `submit.zip` 생성 및 최종 검사

5번 단계에서 참가자가 생성한 `inference.py`, 학습된 모델 및 `requirements.txt`를 압축합니다. 이후 ZIP 내부의 `inference.py`를 다시 읽어 세 함수와 필수 모델 파일을 최종 검사합니다.


In [ ]:
import json
import zipfile
import ast

SUBMIT_PATH = ROOT / "submit.zip"

if not INFERENCE_PATH.is_file():
    raise FileNotFoundError(
        f"추론 파일이 없습니다: {INFERENCE_PATH}. 5번 inference.py 생성 단계를 먼저 실행해 주세요."
    )

inference_source = INFERENCE_PATH.read_text(encoding="utf-8")

# ZIP을 만들기 전에 실제 모듈 최상위에 Stage별 함수가 모두 있는지 검증합니다.
tree = ast.parse(inference_source, filename="inference.py")
defined_functions = {
    node.name for node in tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}
required_functions = {"predict_stage1", "predict_stage2", "predict_stage3"}
missing_functions = sorted(required_functions - defined_functions)
if missing_functions:
    raise RuntimeError(
        "inference.py 생성 실패: 필수 추론 함수가 없습니다: " + str(missing_functions)
    )

if SUBMIT_PATH.exists():
    SUBMIT_PATH.unlink()

with zipfile.ZipFile(SUBMIT_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    archive.writestr("inference.py", inference_source)
    archive.write(ROOT / "requirements.txt", "requirements.txt")
    for model_path in sorted(MODEL_DIR.rglob("*")):
        if model_path.is_file():
            archive.write(model_path, model_path.relative_to(ROOT).as_posix())

with zipfile.ZipFile(SUBMIT_PATH) as archive:
    names = archive.namelist()
    zipped_inference = archive.read("inference.py").decode("utf-8")

required = {
    "inference.py",
    "requirements.txt",
    "model/stage1/best.pt",
    "model/stage2/detector.pth",
    "model/stage3/best.pt",
}
missing = sorted(required - set(names))
if missing:
    raise RuntimeError("제출 ZIP 필수 파일 누락: " + str(missing))

zipped_tree = ast.parse(zipped_inference, filename="submit.zip/inference.py")
zipped_functions = {
    node.name for node in zipped_tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}
missing_in_zip = sorted(required_functions - zipped_functions)
if missing_in_zip:
    raise RuntimeError("ZIP 내부 inference.py 함수 누락: " + str(missing_in_zip))

print(f"생성 완료: {SUBMIT_PATH}")
print(f"압축 크기: {SUBMIT_PATH.stat().st_size / 1024**3:.3f} GB")
print("ZIP 내부 추론 함수:", sorted(required_functions))
print("ZIP 내부 파일:")
for name in names:
    print(" -", name)
